# 16. 형상별 일반화 성능비교

metal shape가 바뀌었을 때 defect segmentation 성능이 변하는지 확인합니다.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch2_utils.py").exists():
    matches = (
        list(Path.cwd().glob("Deeplearning/*/2-1장/ch2_utils.py"))
        + list(Path.cwd().glob("Deeplearning/*/2장/ch2_utils.py"))
        + list(Path.cwd().glob("**/ch2_utils.py"))
    )
    NOTEBOOK_DIR = matches[0].parent if matches else Path("Deeplearning") / "Vision 응용" / "2-1장"
sys.path.append(str(NOTEBOOK_DIR))

from ch2_utils import *

paths = find_paths()
set_korean_font()
set_seed(7)
samples = load_samples(paths.data_root)
paths

In [ ]:
run_dir = paths.runs_root / "baseline_segformer_b0"
sample_metrics, group_metrics, class_metrics = load_run_metrics(run_dir)

## 16-1. Shape별 성능

In [ ]:
shape_table = plot_metric_bar(
    group_metrics,
    grouping="shape_group",
    metric="target_dice_mean",
    title="형상별 defect Dice",
    out_path=paths.runs_root / "16_shape_target_dice.png",
)
display(shape_table)

## 16-2. Shape x Defect heatmap

In [ ]:
pivot = plot_metric_heatmap(
    sample_metrics,
    row="shape_group",
    col="defect_type",
    metric="target_dice",
    title="shape x defect target Dice",
    out_path=paths.runs_root / "16_shape_defect_heatmap.png",
)
display(pivot)

## 16-3. 형상 효과 결론

In [ ]:
print(conclusion_from_group(group_metrics, "shape_group", "target_dice_mean"))
shapes = sorted(sample_metrics["shape_group"].unique())
if len(shapes) >= 2:
    result = compare_two_groups(sample_metrics, "shape_group", shapes[0], shapes[1], "target_dice")
    display(pd.DataFrame([result]))
    print(
        "결론:",
        f"{shapes[0]}와 {shapes[1]}의 차이는 CI 기준으로 관측됩니다." if result["reject_h0_ci_excludes_0"]
        else f"{shapes[0]}와 {shapes[1]}의 차이는 현재 결과만으로 확정하기 어렵습니다."
    )